# Product publish round-trip smoke test

One fast pass over **every** `(sensor, product)` pair in `shared_utils.product_paths.PRODUCT_DIRS`:

    pre-saved fixture -> convert_to_cog (+ activation tags) -> upload to the product's
    canonical S3 key -> read it back -> delete it -> confirm it is gone

No vendor bucket is listed, nothing is downloaded from a data provider, and no `process_*`
CLI runs. Inputs are the committed crops in `tests/fixtures/`, so a full run is minutes,
not hours.

## What this proves

* every product resolves to a destination through `product_paths` (or is reported as an
  undecided path, with the reason);
* `convert_to_cog` produces a COG carrying all six activation tags, under the nodata
  contract that product's dtype implies;
* the canonical prefix `ProgramData/<Sensor>/<Product>/` is actually writable;
* the object round-trips and deletes cleanly, leaving nothing behind;
* all seven workflow notebooks derive their upload key from the same shared table
  (final cell).

## What this does NOT prove

* **the sensors' science.** No `process_landsat89` / `process_capella` / ... is invoked.
  The fixtures are crops of pipeline *outputs*, not vendor *inputs*, so they stand in for
  "a finished product" — the pixels are not what that sensor would really produce.
* **the DPS MAAP workspace-credential path.** `dps/_finalize.sh` publishes via
  `staging_upload.upload_dir_to_staging`, which authenticates through maap-py. No
  notebook uses that path: every workflow notebook uploads with ambient credentials,
  and this notebook does too.

## Safety

`DRY_RUN = True` by default. The destinations are the **live** published prefixes, so a
real run writes next to real products. Three guards:

1. every basename contains `SMOKETEST`, so a leftover object is unmistakable;
2. an existing key is never overwritten — the row fails instead;
3. every key is written to a local manifest *before* it is uploaded, and the **SWEEP** cell
   near the bottom cleans up after an interrupted run.


In [ ]:
# ---- INPUTS ----
# Edit these, then run the notebook top to bottom.

# True  = do everything except the S3 put/delete (no S3 calls at all).
# False = the real round trip against the live staging bucket.
DRY_RUN = True

# Deliberately absurd so a leftover object is obviously not a real product.
# Must still be YYYYMM_Hazard_Location: resolve_metadata splits it into the
# YEAR_MONTH / HAZARD / LOCATION tags.
EVENT_NAME = "209901_Smoketest_DoNotUse"
SOURCE = "SMOKETEST"

# Appears in every basename this notebook creates. The SWEEP cell filters on it,
# so do not remove it from the names.
MARKER = "SMOKETEST"

# None = every sensor in product_paths.SENSOR_DIRS. Or e.g. ["capella", "umbra"].
SENSORS = None

# Local scratch. The COGs and the key manifest are LEFT here after the run, on
# purpose: the manifest is what the SWEEP cell reads to recover from an
# interrupted run, and the COGs are there to inspect when a row fails. Delete the
# directory yourself when you are done with it.
WORK_DIR = "/tmp/product_smoketest"

# ZSTD 22 (the library default) buys nothing on a 256x256 crop and costs seconds
# per product across 40 of them.
COMPRESSION_LEVEL = 1

# Re-download each uploaded object and re-read its tags. Adds a second or two per
# product; it is the only step that proves the bytes that landed are readable.
VERIFY_FROM_S3 = True

import os
import sys
from uuid import uuid4

RUN_ID = uuid4().hex[:8]

# Absolute repo root, so this notebook does not depend on the kernel's cwd.
REPO_ROOT = os.environ.get("DPA_REPO_ROOT")
if not REPO_ROOT:
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "pyproject.toml")):
        _p = os.path.dirname(_p)
    REPO_ROOT = _p
if os.path.join(REPO_ROOT, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

FIXTURE_DIR = os.path.join(REPO_ROOT, "tests", "fixtures")

from shared_utils.product_paths import STAGING_BUCKET

S3_BUCKET = STAGING_BUCKET

print(f"repo root : {REPO_ROOT}")
print(f"fixtures  : {FIXTURE_DIR}")
print(f"bucket    : s3://{S3_BUCKET}")
print(f"run id    : {RUN_ID}")
print(f"DRY_RUN   : {DRY_RUN}" + ("  (no S3 calls)" if DRY_RUN else "  <-- LIVE S3 WRITES"))

In [ ]:
# Auto-populated -- do not edit. (Repo convention: the cell above is what an
# operator edits; this one derives the tag dict that gets baked into every COG.)
from shared_utils import PROCESSOR_STRING

ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}

# The six tags every published COG must carry. YEAR_MONTH / HAZARD / LOCATION are
# split out of ACTIVATION_EVENT by cog_metadata.resolve_metadata at conversion time.
REQUIRED_TAGS = (
    "ACTIVATION_EVENT",
    "YEAR_MONTH",
    "HAZARD",
    "LOCATION",
    "SOURCE",
    "PROCESSOR",
)

for k, v in ACTIVATION_METADATA.items():
    print(f"{k:18s} {v}")

In [ ]:
# ---- PLAN ----
# One row per (sensor, product) in product_paths.PRODUCT_DIRS. Nothing is hardcoded
# here that the table already states: the destination, the local product directory
# and the output filename all come from the same functions the processors call.
import os
from datetime import datetime

from shared_utils import product_paths as pp
from shared_utils.file_naming import create_output_filename, create_sar_output_filename

SAR_SENSORS = {"umbra", "capella", "iceye"}

# A stamp no real acquisition can carry, so a leftover file sorts to the very top
# of a product directory and reads as synthetic at a glance.
SYNTH_DT = datetime(1970, 1, 1, 0, 0, 0)

# Product families, derived from the token sets in product_paths so a rename there
# updates them. The two `|` extras are tokens those sets do not carry: satellogic
# and skysat spell color-infrared "colorir"/"ColorIR", and dNBR is an index.
_INDEX = {t.upper() for t in pp.INDEX_PRODUCT_TOKENS} | {"DNBR"}
_COMPOSITE = {t.upper() for t in pp.COMPOSITE_PRODUCT_TOKENS} | {"COLORIR"}
_CLOUD = {t.upper() for t in pp.CLOUD_MASK_PRODUCT_TOKENS}


def product_family(sensor, product):
    """Classify a product token so the right fixture and nodata value are used."""
    if sensor in SAR_SENSORS:
        return "sar"
    t = product.replace("-", "").replace("_", "").upper()
    if t in _CLOUD:
        return "cloudmask"
    if t == "WATEREXTENT":
        return "waterextent"
    if t == "PANCHROMATIC":
        return "panchromatic"
    if t in _INDEX:
        return "index"
    if t in _COMPOSITE:
        return "composite"
    raise KeyError(
        f"no fixture family for {sensor}/{product}; add it to product_family() "
        f"in this cell"
    )


# family -> (fixture, nodata passed to convert_to_cog, nodata expected on the output).
#
# The nodata column is the point, not decoration: between them these four rows
# exercise every branch of the convert_to_cog(nodata=...) contract --
#   False    declare none AND strip the tag the source carries (8-bit composites)
#   <number> declare exactly that (float32 indices and SAR dB)
#   None     inherit the source tag (float32 panchromatic stand-in)
#   None     ...except for bare 8-bit, where the carve-out strips it (masks)
FAMILY_FIXTURES = {
    "composite":    ("satellogic_truecolor_nodata0_crop.tif", False,   None),
    "index":        ("gaia_atlanta_sample.tif",               -9999.0, -9999.0),
    "panchromatic": ("gaia_atlanta_sample.tif",               None,    -9999.0),
    "cloudmask":    ("cloudmask_byte_nodata255.tif",          None,    None),
    "waterextent":  ("cloudmask_byte_nodata255.tif",          None,    None),
    "sar":          ("umbra_guam_sar_db_crop.tif",            -9999.0, -9999.0),
}

# iceye and capella get their own SAR crop so a per-sensor read problem is not
# masked by every SAR row sharing one file.
SAR_FIXTURE_BY_SENSOR = {
    "iceye": "iceye_guam_sar_db_crop.tif",
    "capella": "iceye_guam_sar_db_crop.tif",
}

SOURCE_DIR = os.path.join(WORK_DIR, "_sources")
os.makedirs(SOURCE_DIR, exist_ok=True)


def output_name(sensor, product):
    """The name this product's processor would write, via the real builders."""
    if sensor in SAR_SENSORS:
        platform = f"{MARKER}-{pp.SENSOR_DIRS[sensor]}-{RUN_ID}"
        return create_sar_output_filename(platform, product, SYNTH_DT, filter_size=5)
    stem = f"{MARKER}_{RUN_ID}_{sensor}_{product}_1970-01-01.tif"
    return create_output_filename(stem, EVENT_NAME)


ROWS = []
for (sensor, product) in sorted(pp.PRODUCT_DIRS):
    if SENSORS is not None and sensor not in SENSORS:
        continue

    row = {
        "sensor": sensor,
        "product": product,
        "status": "PLANNED",
        "detail": "",
        "key": "",
        "local": "",
        "deleted": None,
    }

    try:
        prefix = pp.program_data_prefix(sensor, product)
    except pp.UndecidedProductPath as exc:
        # Not a failure of this notebook -- an open decision in product_paths.
        # Record where the file WOULD land today so the consequence is visible.
        row["status"] = "SKIP"
        row["detail"] = str(exc)
        row["local"] = pp.product_output_dir(WORK_DIR, sensor, product, makedirs=False)
        ROWS.append(row)
        continue

    family = product_family(sensor, product)
    fixture, nodata_in, nodata_out = FAMILY_FIXTURES[family]
    if family == "sar":
        fixture = SAR_FIXTURE_BY_SENSOR.get(sensor, fixture)

    name = output_name(sensor, product)
    row.update(
        family=family,
        fixture=os.path.join(FIXTURE_DIR, fixture),
        nodata_in=nodata_in,
        nodata_out=nodata_out,
        prefix=prefix,
        name=name,
        key=f"{prefix}/{name}",
        local=os.path.join(pp.product_output_dir(WORK_DIR, sensor, product), name),
    )
    ROWS.append(row)

# A loop that reports success over zero items is the failure mode this repo has
# actually shipped. Refuse to continue on an empty plan.
assert ROWS, f"no products planned (SENSORS={SENSORS!r})"

planned = [r for r in ROWS if r["status"] == "PLANNED"]
skipped = [r for r in ROWS if r["status"] == "SKIP"]

print(f"{len(planned)} product(s) planned, {len(skipped)} skipped\n")
print(f"{'sensor':<11} {'product':<18} {'family':<13} destination key")
print("-" * 118)
for r in planned:
    print(f"{r['sensor']:<11} {r['product']:<18} {r['family']:<13} {r['key']}")
for r in skipped:
    print(f"{r['sensor']:<11} {r['product']:<18} {'-':<13} SKIP: undecided destination")

In [ ]:
# ---- PREFLIGHT ----
# Prove the destinations are writable BEFORE converting anything. Grants on this
# bucket are per-prefix, and head_bucket succeeds for a read-only identity, so the
# probe has to be a real PutObject under each prefix (can_write_to_bucket does
# exactly that, then deletes its probe).
import boto3

from shared_utils.s3_operations import can_write_to_bucket

s3 = boto3.client("s3")

PREFIXES = sorted({r["prefix"] for r in ROWS if r["status"] == "PLANNED"})

if DRY_RUN:
    print(f"DRY_RUN -- skipping the write preflight for {len(PREFIXES)} prefix(es).")
    for p in PREFIXES:
        print(f"  would probe s3://{S3_BUCKET}/{p}/")
else:
    unwritable = []
    for p in PREFIXES:
        ok, detail = can_write_to_bucket(s3, S3_BUCKET, p, verbose=False)
        print(("  OK        " if ok else "  NOT OK    ") + f"s3://{S3_BUCKET}/{p}/"
              + ("" if ok else f"\n            {detail}"))
        if not ok:
            unwritable.append((p, detail))
    if unwritable:
        raise RuntimeError(
            f"{len(unwritable)} of {len(PREFIXES)} destination prefixes are not "
            f"writable with these credentials; nothing was converted. First: "
            f"{unwritable[0][0]} -- {unwritable[0][1]}"
        )
    print(f"\nAll {len(PREFIXES)} destination prefix(es) writable.")

In [ ]:
# ---- ROUND TRIP ----
# Per product: convert + tag, verify locally, upload, verify from S3, delete,
# confirm gone. One row's failure never aborts the rest, and an exception after
# the upload still deletes the object.
import json
import os
import shutil
import tempfile

import rasterio
from botocore.exceptions import ClientError

from shared_utils.cog_utils import convert_to_cog
from shared_utils.s3_operations import check_s3_file_exists
from shared_utils.s3utils import upload_file_to_s3

MANIFEST = os.path.join(WORK_DIR, f"manifest_{RUN_ID}.json")


def _record(key):
    """Append a key to the on-disk manifest BEFORE it is uploaded, so an
    interrupted kernel is still recoverable by the SWEEP cell."""
    keys = []
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as fh:
            keys = json.load(fh)
    keys.append(key)
    with open(MANIFEST, "w") as fh:
        json.dump(keys, fh, indent=2)


def _check_tags(path):
    with rasterio.open(path) as src:
        tags = src.tags()
        nodata = src.nodata
    missing = [t for t in REQUIRED_TAGS if not tags.get(t)]
    if missing:
        raise AssertionError(f"missing tag(s): {', '.join(missing)}")
    if tags["ACTIVATION_EVENT"] != EVENT_NAME:
        raise AssertionError(
            f"ACTIVATION_EVENT is {tags['ACTIVATION_EVENT']!r}, expected {EVENT_NAME!r}"
        )
    return nodata


for r in ROWS:
    if r["status"] != "PLANNED":
        continue

    uploaded = False
    try:
        # 1. stage the fixture outside the product tree, so a recursive glob of
        #    the product directories sees products only.
        src = os.path.join(SOURCE_DIR, f"{r['sensor']}_{r['product']}_src.tif")
        shutil.copyfile(r["fixture"], src)

        # 2. convert + embed the activation tags. dst_crs=None keeps the native
        #    projection (no warp) -- this is a path/plumbing test, not a repro-
        #    jection test, and the warp is the slow part.
        convert_to_cog(
            src,
            output_cog=r["local"],
            nodata=r["nodata_in"],
            dst_crs=None,
            compression="ZSTD",
            compression_level=COMPRESSION_LEVEL,
            metadata=ACTIVATION_METADATA,
            quiet=True,
        )

        # 3. local verification: tags present, and the nodata contract resolved
        #    the way this product's dtype implies.
        nodata = _check_tags(r["local"])
        if nodata != r["nodata_out"]:
            raise AssertionError(
                f"nodata is {nodata!r}, expected {r['nodata_out']!r} "
                f"(passed nodata={r['nodata_in']!r})"
            )
        local_size = os.path.getsize(r["local"])

        if DRY_RUN:
            r["status"] = "DRY"
            r["detail"] = f"COG ok ({local_size:,} B), nodata={nodata!r}; no S3 calls"
        else:
            # 4. never overwrite. A real product at this key means the synthetic
            #    name collided with something -- stop, do not touch it.
            if check_s3_file_exists(s3, S3_BUCKET, r["key"]):
                raise AssertionError(
                    f"s3://{S3_BUCKET}/{r['key']} already exists; refusing to overwrite"
                )

            # 5. upload through the same helper the workflow notebooks call.
            _record(r["key"])
            upload_file_to_s3(r["local"], f"s3://{S3_BUCKET}/{r['key']}")
            uploaded = True

            head = s3.head_object(Bucket=S3_BUCKET, Key=r["key"])
            if head["ContentLength"] != local_size:
                raise AssertionError(
                    f"uploaded {local_size} B but S3 reports {head['ContentLength']} B"
                )

            # 6. read the bytes back off S3 and re-check the tags.
            if VERIFY_FROM_S3:
                with tempfile.TemporaryDirectory() as td:
                    back = os.path.join(td, os.path.basename(r["key"]))
                    s3.download_file(S3_BUCKET, r["key"], back)
                    _check_tags(back)

            r["status"] = "PASS"
            r["detail"] = f"{local_size:,} B, nodata={nodata!r}"

    except Exception as exc:
        r["status"] = "FAIL"
        r["detail"] = f"{type(exc).__name__}: {exc}"

    finally:
        # 7. delete whatever reached S3, even if a later step raised.
        if uploaded:
            try:
                s3.delete_object(Bucket=S3_BUCKET, Key=r["key"])
                try:
                    s3.head_object(Bucket=S3_BUCKET, Key=r["key"])
                    r["deleted"] = False
                    r["status"] = "FAIL"
                    r["detail"] = "object still present after delete_object"
                except ClientError as exc:
                    if exc.response["Error"]["Code"] in ("404", "NoSuchKey"):
                        r["deleted"] = True
                    else:
                        r["deleted"] = False
                        r["status"] = "FAIL"
                        r["detail"] = f"delete unconfirmed: {exc}"
            except Exception as exc:
                r["deleted"] = False
                r["status"] = "FAIL"
                r["detail"] = f"delete failed: {type(exc).__name__}: {exc}"

    print(f"  {r['status']:<5} {r['sensor']}/{r['product']}"
          + (f" -- {r['detail']}" if r["status"] in ("FAIL",) else ""))

print("\ndone")

In [ ]:
# ---- RESULTS ----
# A gate, not a report: anything that failed, or anything left on S3, raises.
from collections import Counter

counts = Counter(r["status"] for r in ROWS)

print(f"{'sensor':<11} {'product':<18} {'status':<7} {'del':<5} destination key")
print("-" * 130)
for r in ROWS:
    deleted = "-" if r["deleted"] is None else ("yes" if r["deleted"] else "NO")
    print(f"{r['sensor']:<11} {r['product']:<18} {r['status']:<7} {deleted:<5} "
          f"{r['key'] or '(no destination)'}")

print()
for status in ("PASS", "DRY", "SKIP", "FAIL"):
    if counts.get(status):
        print(f"  {status:<5} {counts[status]}")

failed = [r for r in ROWS if r["status"] == "FAIL"]
if failed:
    for r in failed:
        print(f"\nFAIL {r['sensor']}/{r['product']}\n     {r['detail']}")

left_behind = [r for r in ROWS if r["deleted"] is False]

skipped = [r for r in ROWS if r["status"] == "SKIP"]
if skipped:
    print("\nSkipped -- these products have no decided S3 destination "
          "(shared_utils/product_paths.py):")
    for r in skipped:
        print(f"\n  {r['sensor']}/{r['product']}")
        print(f"    would be written to: {r['local']}")
        print(f"    {r['detail']}")

if failed or left_behind:
    raise RuntimeError(
        f"{len(failed)} product(s) failed, {len(left_behind)} object(s) left on S3. "
        f"Run the SWEEP cell below to clean up."
    )

print("\nOK -- every planned product converted, tagged"
      + ("" if DRY_RUN else ", published, verified and deleted") + ".")

## Cleanup

Run this only if a run was interrupted, or if the results cell reported objects left behind.

In [ ]:
# ---- SWEEP ----
# Standalone cleanup. Safe to run at any time, including after an interrupted run
# (it reads every manifest this notebook has written, and also lists the live
# product prefixes). It only ever considers keys whose BASENAME contains MARKER,
# so it cannot touch a real product.
import glob
import json
import os

from shared_utils import product_paths as pp

SWEEP_CONFIRM = False   # set True to actually delete what is listed

candidates = set()

# 1. keys this notebook recorded before uploading them
for m in sorted(glob.glob(os.path.join(WORK_DIR, "manifest_*.json"))):
    with open(m) as fh:
        candidates.update(json.load(fh))

# 2. anything marked sitting in a live product prefix
prefixes = set()
for (sensor, product), value in pp.PRODUCT_DIRS.items():
    if value is not None:
        prefixes.add(pp.program_data_prefix(sensor, product))

paginator = s3.get_paginator("list_objects_v2")
for prefix in sorted(prefixes):
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=f"{prefix}/"):
        for obj in page.get("Contents", []):
            if MARKER in os.path.basename(obj["Key"]):
                candidates.add(obj["Key"])

# Only keys that actually exist right now, and only marked ones.
orphans = []
for key in sorted(candidates):
    if MARKER not in os.path.basename(key):
        continue
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        orphans.append(key)
    except Exception:
        pass

if not orphans:
    print(f"No {MARKER} objects in s3://{S3_BUCKET}/{pp.PROGRAM_DATA_ROOT}/. Nothing to sweep.")
else:
    print(f"{len(orphans)} {MARKER} object(s) present:")
    for key in orphans:
        print(f"  s3://{S3_BUCKET}/{key}")
    if SWEEP_CONFIRM:
        for key in orphans:
            s3.delete_object(Bucket=S3_BUCKET, Key=key)
            print(f"  deleted {key}")
        print(f"\nDeleted {len(orphans)} object(s).")
    else:
        print("\nSWEEP_CONFIRM is False -- nothing deleted. "
              "Set it True and re-run this cell to delete the keys above.")

## Do the workflow notebooks use the table?

Independent of everything above, and safe to run on its own.

In [ ]:
# ---- NOTEBOOK PATH CHECK ----
# The loop above proves the product_paths table works. This proves the seven
# workflow notebooks actually USE it -- a notebook that hardcodes a folder name
# is the real "published to the wrong path" failure, and it is invisible to
# everything else here.
import glob
import io
import os
import re
import tokenize

import nbformat

from shared_utils import product_paths as pp

# A literal ProgramData/ path with a REAL sensor directory in it, NOT followed by
# a "<...>" placeholder. Both halves are needed to avoid false positives: every
# notebook prints "ProgramData/<Sensor>/<Product>/" as a human-readable
# placeholder, and sentinel2_odr spells its own sensor out in that print
# ("ProgramData/Sentinel-2/<Product>/") while still resolving the real key
# through product_dir().
HARDCODED = re.compile(
    r"ProgramData/(" + "|".join(re.escape(d) for d in pp.SENSOR_DIRS.values()) + r")/(?!<)"
)


def strip_comments(src):
    """Blank out comment text so a path in a COMMENT is not read as a hardcode."""
    lines = src.splitlines()
    try:
        toks = list(tokenize.generate_tokens(io.StringIO(src).readline))
    except (tokenize.TokenError, IndentationError, SyntaxError):
        return src
    for tok in toks:
        if tok.type == tokenize.COMMENT:
            row, col = tok.start
            lines[row - 1] = lines[row - 1][:col]
    return "\n".join(lines)

# Any one of these means the key came out of the shared table. Matched on a word
# boundary so prefix_for_product_dir() is not also reported as product_dir().
RESOLVERS = ("prefix_for_product_dir", "product_dir", "upload_dir_ambient")

nb_paths = sorted(glob.glob(os.path.join(REPO_ROOT, "notebooks", "*.ipynb")))
problems = []

print(f"{'notebook':<32} {'resolver(s)':<42} {'bucket':<16} verdict")
print("-" * 108)

for path in nb_paths:
    nb = nbformat.read(path, as_version=4)
    cells = [c.source for c in nb.cells if c.cell_type == "code"]
    whole = "\n".join(cells)
    upload = [c for c in cells if "UPLOAD TO S3" in c]
    name = os.path.basename(path)

    if len(upload) != 1:
        problems.append(f"{name}: expected 1 'UPLOAD TO S3' cell, found {len(upload)}")
        print(f"{name:<32} {'-':<42} {'-':<16} NO UPLOAD CELL")
        continue

    cell = upload[0]
    used = [r for r in RESOLVERS if re.search(r"\b" + r + r"\(", cell)]
    bucket_ok = "STAGING_BUCKET" in whole
    hard = HARDCODED.search(strip_comments(cell))

    verdict = "ok"
    if not used:
        problems.append(f"{name}: upload cell calls none of {RESOLVERS}")
        verdict = "NO TABLE LOOKUP"
    if not bucket_ok:
        problems.append(f"{name}: does not take its bucket from product_paths.STAGING_BUCKET")
        verdict = "BUCKET NOT FROM TABLE"
    if hard:
        problems.append(f"{name}: hardcodes {hard.group(0)!r} in the upload cell")
        verdict = f"HARDCODED {hard.group(0)}"

    print(f"{name:<32} {(', '.join(used) if used else '-'):<42} "
          f"{('STAGING_BUCKET' if bucket_ok else '-'):<16} {verdict}")

print()
if problems:
    for p in problems:
        print(f"  {p}")
    raise RuntimeError(f"{len(problems)} notebook(s) do not resolve their destination "
                       f"through shared_utils.product_paths")
print(f"All {len(nb_paths)} workflow notebooks resolve their destination through "
      f"shared_utils.product_paths.")